# A vs B/C Binary Model Predictions

This notebook loads the final binary classifier for the first stage of the hierarchical strategy and uses it to generate predictions for:

1. the labeled test set, for evaluation and validation;
2. all unlabeled HCPs, for business use.

The model predicts whether each HCP should be classified as:

- `SEG_A`
- `SEG_B/C`

The selected decision threshold is loaded from the model metadata, so the notebook can be reused without retraining.

## 1. Setup

This section imports the required libraries and defines the local paths used throughout the notebook.

Update the paths below if the folder structure changes.

In [1]:
import json
import shutil
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    roc_auc_score,
)

In [2]:
# Local directories
MODEL_DIR = Path(r"C:\Users\omarl\Downloads\pfizer_models")
TENSOR_DIR = Path(r"C:\Users\omarl\Downloads\pfizer_tensors")
OUTPUT_DIR = Path(r"C:\Users\omarl\Downloads\pfizer_outputs")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Model artifacts
MODEL_PATH = MODEL_DIR / "best_binary_segA_vs_segBC.joblib"
METADATA_PATH = MODEL_DIR / "model_metadata.json"

# Tensor artifacts
X_PATH = TENSOR_DIR / "X_features.pt"
Y_PATH = TENSOR_DIR / "y_labels.pt"
FOLDS_PATH = TENSOR_DIR / "folds.pt"
MANIFEST_PATH = TENSOR_DIR / "tensor_manifest.csv"

# Evaluation setup
TEST_FOLD = 3
TARGET_NAMES_BINARY = ["SEG_A", "SEG_BC"]

## 2. Load Final Model and Metadata

The `.joblib` file contains the trained model.  
The metadata file contains the selected threshold, model name, label mapping, and evaluation context.

The threshold is important because the model outputs probabilities, and the final binary label is assigned using:

`SEG_B/C if prob_SEG_BC >= threshold, otherwise SEG_A`

In [3]:
best_model = joblib.load(MODEL_PATH)

with open(METADATA_PATH, "r") as f:
    metadata = json.load(f)

metadata

{'best_model_name': 'HistGradientBoosting',
 'best_threshold': 0.45,
 'task': 'SEG_A vs SEG_BC',
 'label_mapping': {'0': 'SEG_A', '1': 'SEG_BC'},
 'test_metrics': {'model': 'HistGradientBoosting',
  'threshold': 0.45,
  'accuracy': 0.7722689075630252,
  'macro_f1': 0.7703775750550399,
  'weighted_f1': 0.7719711978163979,
  'roc_auc': 0.8518300292864351,
  'SEG_A_precision': 0.7809885931558935,
  'SEG_A_recall': 0.8017174082747853,
  'SEG_A_f1': 0.7912172573189522,
  'SEG_BC_precision': 0.7615023474178404,
  'SEG_BC_recall': 0.737943585077343,
  'SEG_BC_f1': 0.7495378927911276}}

In [4]:
best_threshold = metadata["best_threshold"]
best_model_name = metadata["best_model_name"]

print("Loaded model:", best_model_name)
print("Decision threshold:", best_threshold)

Loaded model: HistGradientBoosting
Decision threshold: 0.45


## 3. Load Tensors and Manifest

The tensors contain the modeling features, labels, and fold assignments.

Expected tensor format:

`X = (HCPs, weeks, features)`

The model was trained on flattened temporal tensors, so each HCP is later transformed into:

`weeks × features`

In [5]:
X_torch = torch.load(X_PATH)
y_torch = torch.load(Y_PATH)
fold_torch = torch.load(FOLDS_PATH)

X_tf = tf.convert_to_tensor(X_torch.cpu().numpy(), dtype=tf.float32)
y_tf = tf.convert_to_tensor(y_torch.cpu().numpy(), dtype=tf.int64)
fold_tf = tf.convert_to_tensor(fold_torch.cpu().numpy(), dtype=tf.int64)

manifest_df = pd.read_csv(MANIFEST_PATH)

print("X:", X_tf.shape)
print("y:", y_tf.shape)
print("folds:", fold_tf.shape)
print("manifest:", manifest_df.shape)

assert len(manifest_df) == X_tf.shape[0], "Manifest and tensor rows are not aligned."
manifest_df.head()

C:\Users\omarl\AppData\Local\Temp\ipykernel_16852\1405552982.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  X_torch = torch.load(X_PATH)
C:\Users\omarl\AppData\Local\Te

X: (20931, 86, 65)
y: (20931,)
folds: (20931,)
manifest: (20931, 3)


,NUEVO_ID,ATSEG_HCP,HCP_FOLD
0,1,NaN,-1
1,2,NaN,-1
2,3,SEG_A,1
3,4,NaN,-1
4,5,NaN,-1


## 4. Helper Functions

These functions keep prediction logic consistent for labeled and unlabeled HCPs.

In [6]:
def flatten_tensor(X_tensor):
    """Convert a 3D temporal tensor into a 2D matrix for sklearn-style models."""
    return X_tensor.numpy().reshape(X_tensor.shape[0], -1)


def binary_label_from_probability(prob_seg_bc, threshold):
    """Apply the selected decision threshold to obtain the final binary class."""
    pred_encoded = (prob_seg_bc >= threshold).astype(int)
    pred_label = np.where(pred_encoded == 0, "SEG_A", "SEG_BC")
    return pred_encoded, pred_label


def map_original_label(y_encoded):
    """Map original 3-class encoded labels into readable labels."""
    return np.where(
        y_encoded == 0,
        "SEG_A",
        np.where(y_encoded == 1, "SEG_B", np.where(y_encoded == 2, "SEG_C", "UNLABELED")),
    )


def map_binary_label(y_binary):
    """Map binary encoded labels into readable labels."""
    return np.where(y_binary == 0, "SEG_A", "SEG_BC")

## 5. Evaluate on Labeled Test Set

This section evaluates the loaded model on the labeled test fold.  
This is useful to confirm that the saved model and threshold reproduce the expected performance.

In [7]:
# Keep only labeled HCPs.
labeled_mask = y_tf != -1

X_labeled = tf.boolean_mask(X_tf, labeled_mask)
y_labeled = tf.boolean_mask(y_tf, labeled_mask)
fold_labeled = tf.boolean_mask(fold_tf, labeled_mask)

manifest_labeled = manifest_df.loc[labeled_mask.numpy()].reset_index(drop=True)

print("X labeled:", X_labeled.shape)
print("y labeled:", y_labeled.shape)
print("manifest labeled:", manifest_labeled.shape)

X labeled: (11899, 86, 65)
y labeled: (11899,)
manifest labeled: (11899, 3)


In [8]:
# Select test fold.
test_mask = fold_labeled == TEST_FOLD

X_test = tf.boolean_mask(X_labeled, test_mask)
y_test_original = tf.boolean_mask(y_labeled, test_mask).numpy()
fold_test = tf.boolean_mask(fold_labeled, test_mask).numpy()

manifest_test = manifest_labeled.loc[test_mask.numpy()].reset_index(drop=True)

# Convert original 3-class labels into the binary target:
# 0 = SEG_A, 1 = SEG_B/C
y_test_binary = (y_test_original != 0).astype(int)

print("X_test:", X_test.shape)
print("Test labels:")
print(pd.Series(y_test_binary).value_counts().sort_index())

X_test: (2380, 86, 65)
Test labels:
0    1281
1    1099
Name: count, dtype: int64


In [9]:
# Generate test predictions.
X_test_flat = flatten_tensor(X_test)

test_proba_bc = best_model.predict_proba(X_test_flat)[:, 1]
test_pred_binary, test_pred_label = binary_label_from_probability(
    test_proba_bc,
    best_threshold,
)

true_original_label = map_original_label(y_test_original)
true_binary_label = map_binary_label(y_test_binary)

test_predictions_df = pd.DataFrame({
    "NUEVO_ID": manifest_test["NUEVO_ID"].values if "NUEVO_ID" in manifest_test.columns else np.arange(len(manifest_test)),
    "true_original_label": true_original_label,
    "true_original_encoded": y_test_original,
    "true_binary_label": true_binary_label,
    "true_binary_encoded": y_test_binary,
    "prob_SEG_A": 1 - test_proba_bc,
    "prob_SEG_BC": test_proba_bc,
    "pred_binary_label": test_pred_label,
    "pred_binary_encoded": test_pred_binary,
    "decision_threshold": best_threshold,
    "hcp_fold": fold_test,
    "model_name": best_model_name,
})

test_predictions_df.head()

,NUEVO_ID,true_original_label,true_original_encoded,true_binary_label,true_binary_encoded,prob_SEG_A,prob_SEG_BC,pred_binary_label,pred_binary_encoded,decision_threshold,hcp_fold,model_name
0,20,SEG_A,0,SEG_A,0,0.843814,0.156186,SEG_A,0,0.45,3,HistGradientBoosting
1,47,SEG_A,0,SEG_A,0,0.643654,0.356346,SEG_A,0,0.45,3,HistGradientBoosting
2,63,SEG_A,0,SEG_A,0,0.774528,0.225472,SEG_A,0,0.45,3,HistGradientBoosting
3,69,SEG_B,1,SEG_BC,1,0.062458,0.937542,SEG_BC,1,0.45,3,HistGradientBoosting
4,76,SEG_A,0,SEG_A,0,0.900509,0.099491,SEG_A,0,0.45,3,HistGradientBoosting


In [10]:
# Evaluate test predictions.
print("Accuracy:", accuracy_score(y_test_binary, test_pred_binary))
print("Macro F1:", f1_score(y_test_binary, test_pred_binary, average="macro"))
print("Weighted F1:", f1_score(y_test_binary, test_pred_binary, average="weighted"))
print("ROC AUC:", roc_auc_score(y_test_binary, test_proba_bc))

print()
print(classification_report(
    y_test_binary,
    test_pred_binary,
    target_names=TARGET_NAMES_BINARY,
))

Accuracy: 0.7722689075630252
Macro F1: 0.7703775750550399
Weighted F1: 0.7719711978163979
ROC AUC: 0.8518300292864351

              precision    recall  f1-score   support

       SEG_A       0.78      0.80      0.79      1281
      SEG_BC       0.76      0.74      0.75      1099

    accuracy                           0.77      2380
   macro avg       0.77      0.77      0.77      2380
weighted avg       0.77      0.77      0.77      2380



In [11]:
# Save labeled test predictions.
test_output_path = OUTPUT_DIR / "test_predictions_binary_segA_vs_segBC_with_hcp_id.csv"

test_predictions_df.to_csv(test_output_path, index=False)

print("Saved test predictions to:", test_output_path)

Saved test predictions to: C:\Users\omarl\Downloads\pfizer_outputs\test_predictions_binary_segA_vs_segBC_with_hcp_id.csv


## 6. Generate Predictions for All Unlabeled HCPs

This is the main business output of the notebook.  
All HCPs with `y = -1` are treated as unlabeled and scored using the final binary model.

In [12]:
# Select all unlabeled HCPs.
unlabeled_mask = y_tf == -1

X_unlabeled = tf.boolean_mask(X_tf, unlabeled_mask)
y_unlabeled = tf.boolean_mask(y_tf, unlabeled_mask).numpy()
fold_unlabeled = tf.boolean_mask(fold_tf, unlabeled_mask).numpy()

manifest_unlabeled = manifest_df.loc[unlabeled_mask.numpy()].reset_index(drop=True)

print("X_unlabeled:", X_unlabeled.shape)
print("y_unlabeled:", y_unlabeled.shape)
print("manifest_unlabeled:", manifest_unlabeled.shape)

X_unlabeled: (9032, 86, 65)
y_unlabeled: (9032,)
manifest_unlabeled: (9032, 3)


In [13]:
# Generate unlabeled predictions.
X_unlabeled_flat = flatten_tensor(X_unlabeled)

unlabeled_proba_bc = best_model.predict_proba(X_unlabeled_flat)[:, 1]
unlabeled_pred_binary, unlabeled_pred_label = binary_label_from_probability(
    unlabeled_proba_bc,
    best_threshold,
)

unlabeled_predictions_df = pd.DataFrame({
    "NUEVO_ID": manifest_unlabeled["NUEVO_ID"].values if "NUEVO_ID" in manifest_unlabeled.columns else np.arange(len(manifest_unlabeled)),
    "true_label_encoded": y_unlabeled,  # should be -1
    "true_label": "UNLABELED",
    "prob_SEG_A": 1 - unlabeled_proba_bc,
    "prob_SEG_BC": unlabeled_proba_bc,
    "pred_binary_encoded": unlabeled_pred_binary,
    "pred_binary_label": unlabeled_pred_label,
    "decision_threshold": best_threshold,
    "hcp_fold": fold_unlabeled,
    "model_name": best_model_name,
})

# Add optional manifest columns if available.
for col in ["ATSEG_HCP", "HCP_FOLD"]:
    if col in manifest_unlabeled.columns and col not in unlabeled_predictions_df.columns:
        unlabeled_predictions_df.insert(
            1,
            col,
            manifest_unlabeled[col].values,
        )

unlabeled_predictions_df.head()

,NUEVO_ID,HCP_FOLD,ATSEG_HCP,true_label_encoded,true_label,prob_SEG_A,prob_SEG_BC,pred_binary_encoded,pred_binary_label,decision_threshold,hcp_fold,model_name
0,1,-1,NaN,-1,UNLABELED,0.889189,0.110811,0,SEG_A,0.45,-1,HistGradientBoosting
1,2,-1,NaN,-1,UNLABELED,0.909169,0.090831,0,SEG_A,0.45,-1,HistGradientBoosting
2,4,-1,NaN,-1,UNLABELED,0.910450,0.089550,0,SEG_A,0.45,-1,HistGradientBoosting
3,5,-1,NaN,-1,UNLABELED,0.780160,0.219840,0,SEG_A,0.45,-1,HistGradientBoosting
4,6,-1,NaN,-1,UNLABELED,0.932220,0.067780,0,SEG_A,0.45,-1,HistGradientBoosting


### Prediction Distribution

The following outputs summarize how the model classified the unlabeled population and how confident those predictions were.

In [14]:
print("Prediction counts:")
display(unlabeled_predictions_df["pred_binary_label"].value_counts())

print("Prediction percentages:")
display((unlabeled_predictions_df["pred_binary_label"].value_counts(normalize=True) * 100).round(2))

print("Probability summary:")
display(unlabeled_predictions_df[["prob_SEG_A", "prob_SEG_BC"]].describe())

Prediction counts:


pred_binary_label
SEG_A     8399
SEG_BC     633
Name: count, dtype: int64

Prediction percentages:


pred_binary_label
SEG_A     92.99
SEG_BC     7.01
Name: proportion, dtype: float64

Probability summary:


,prob_SEG_A,prob_SEG_BC
count,9032.000000,9032.000000
mean,0.840028,0.159972
std,0.177819,0.177819
min,0.001723,0.031565
25%,0.856335,0.076391
50%,0.904689,0.095311
75%,0.923609,0.143665
max,0.968435,0.998277


In [15]:
# Save unlabeled predictions.
unlabeled_output_path = OUTPUT_DIR / "unlabeled_predictions_binary_segA_vs_segBC_with_hcp_id.csv"

unlabeled_predictions_df.to_csv(unlabeled_output_path, index=False)

print("Saved unlabeled predictions to:", unlabeled_output_path)

Saved unlabeled predictions to: C:\Users\omarl\Downloads\pfizer_outputs\unlabeled_predictions_binary_segA_vs_segBC_with_hcp_id.csv


## 7. Prepare Hugging Face Upload Folder

This section collects the model, metadata, and prediction outputs into one clean folder that can be uploaded to Hugging Face.

Only run this section if you need to update the shared model repository.

In [ ]:
HF_EXPORT_DIR = MODEL_DIR / "hf_binary_segA_vs_segBC"
HF_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

files_to_copy = [
    MODEL_PATH,
    METADATA_PATH,
    MODEL_DIR / "binary_model_threshold_comparison_validation.csv",
    test_output_path,
    unlabeled_output_path,
]

for file_path in files_to_copy:
    if file_path.exists():
        shutil.copy(file_path, HF_EXPORT_DIR / file_path.name)
        print("Copied:", file_path.name)
    else:
        print("Missing optional file:", file_path)

## 8. Create Model Card

The README summarizes the model task, decision rule, files, and reuse instructions.

In [18]:
readme_text = f"""
---
license: other
tags:
- binary-classification
- healthcare
- physician-segmentation
- scikit-learn
- xgboost
---

# Binary SEG_A vs SEG_B/C Classifier

This repository contains the selected binary classifier for the first stage of a hierarchical physician segmentation strategy.

## Task

Binary classification:

- `0`: SEG_A
- `1`: SEG_B/C

The model predicts whether a physician belongs to `SEG_A` or should be routed to the second-stage `SEG_B` vs `SEG_C` classifier.

## Selected Model

Best model: `{best_model_name}`  
Decision threshold for SEG_B/C: `{best_threshold}`

## Files

- `best_binary_segA_vs_segBC.joblib`: trained model
- `model_metadata.json`: model configuration and selected threshold
- `binary_model_threshold_comparison_validation.csv`: validation threshold comparison
- `test_predictions_binary_segA_vs_segBC_with_hcp_id.csv`: test-set predictions with HCP ID

## Notes

The model uses flattened temporal tensors as input. Each physician is represented by weekly behavior across multiple features.

The prediction probability `prob_SEG_BC` can be used to decide whether a physician should be classified as `SEG_A` or passed to the next B/C decision model.
"""

with open(HF_EXPORT_DIR / "README.md", "w", encoding="utf-8") as f:
    f.write(readme_text)

## 9. Upload to Hugging Face

Run this section after logging in with a Hugging Face token that has write permissions.

If the repo contains HCP IDs or prediction outputs, keep it private.

In [ ]:
from huggingface_hub import login, create_repo, upload_folder, whoami

login()

In [ ]:
# Set repository information.
# If you are logged in, this automatically uses your Hugging Face username.
HF_USERNAME = whoami()["name"]
REPO_NAME = "pfizer-binary-segA-vs-segBC"
REPO_ID = f"{HF_USERNAME}/{REPO_NAME}"

PRIVATE_REPO = True

print("Repo ID:", REPO_ID)
print("Private:", PRIVATE_REPO)

In [ ]:
create_repo(
    repo_id=REPO_ID,
    repo_type="model",
    private=PRIVATE_REPO,
    exist_ok=True,
)

upload_folder(
    repo_id=REPO_ID,
    repo_type="model",
    folder_path=str(HF_EXPORT_DIR),
    commit_message="Upload binary SEG_A vs SEG_BC classifier and predictions",
)

print(f"Uploaded to: https://huggingface.co/{REPO_ID}")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

https://huggingface.co/pfizer-project-team/binary-segA-vs-segBC


## 10. Summary

This notebook:

1. loaded the final binary `SEG_A` vs `SEG_B/C` model;
2. evaluated it on the labeled test fold;
3. generated predictions for all unlabeled HCPs;
4. exported prediction CSVs with HCP IDs;
5. prepared and uploaded the model artifacts to Hugging Face.

These outputs support the first stage of the hierarchical segmentation pipeline.